<a href="https://colab.research.google.com/github/niikun/ezkl/blob/main/set_membership.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set Membership

In [52]:
try:
    # install ezkl
    import google.colab
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ezkl"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "onnx"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pytest"])

# rely on local installation of ezkl if the notebook is not in colab
except:
    pass

In [53]:
import logging
FORMAT = '%(levelname)s %(name)s %(asctime)-15s %(filename)s:%(lineno)d %(message)s'
logging.basicConfig(format=FORMAT)
logging.getLogger().setLevel(logging.DEBUG)

## 単純な場合　要素が数字

In [54]:
from torch import nn
import ezkl
import os
import json
import torch

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel,self).__init__()

    def forward(self,x,y):
        diff = (x-y)
        membership_test = torch.prod(diff, dim=1)
        return (membership_test,y)

circuit = MyModel()

In [55]:
model_path = os.path.join('network.onnx')
compiled_model_path = os.path.join('network.compiled')
pk_path = os.path.join('test.pk')
vk_path = os.path.join('test.vk')
settings_path = os.path.join('settings.json')

witness_path = os.path.join('witness.json')
data_path = os.path.join('input.json')

In [56]:
x = torch.zeros(1,*[1], requires_grad=True)
y = torch.tensor([0.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0], requires_grad=True)

y_input = [0.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0]

result = []

for e in y_input:
    print(ezkl.float_to_felt(e,7))
    result.append(ezkl.poseidon_hash([ezkl.float_to_felt(e, 7)])[0])

y = y.unsqueeze(0)
y = y.reshape(1,9)

circuit.eval()

torch.onnx.export(circuit,
                  (x,y),
                  model_path,
                  export_params=True,
                  opset_version=14,
                  do_constant_folding=True,
                  input_names = ['input'],
                  output_names = ['output'],
                  dynamic_axes={'input' : {0 : 'batch_size'},
                                'output' : {0 : 'batch_size'}},
                  dynamo=False
                  )

data_array_x = ((x).detach().numpy()).reshape([-1]).tolist()
data_array_y = result
print(data_array_y)

data = dict(input_data = [data_array_x, data_array_y])

print(data)

    # Serialize data into file:
json.dump( data, open(data_path, 'w' ))

0000000000000000000000000000000000000000000000000000000000000000
0001000000000000000000000000000000000000000000000000000000000000
8001000000000000000000000000000000000000000000000000000000000000
0002000000000000000000000000000000000000000000000000000000000000
8002000000000000000000000000000000000000000000000000000000000000
0003000000000000000000000000000000000000000000000000000000000000
8003000000000000000000000000000000000000000000000000000000000000
0004000000000000000000000000000000000000000000000000000000000000
8004000000000000000000000000000000000000000000000000000000000000
['c9ee727ef8f6307295e9e69831ecddb7866fa7c072dc626ad45a803258c2d411', '59a711784a1a751ed727c22a8ffb8f9f27171bd2d8f57fca2ba9ebf3dac3f023', '2ad51066722d0ce27aa8b16c8da8b229d009e8f804a49771309b3347f23ff504', '5f8c61fda484271ae4b14dd122a938ba74fa1cf3b1a4483ba4253b77c87a920a', 'a761d7c193856b47c39447bdd4a2a85fca8aebc604d8555299e01a5b5c39cc2c', '28c2facf747bda1cce78c265233ac09dda1708937c90af3b761ca579c248010d', 'e8670

/tmp/ipykernel_33816/3521457528.py:17: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(circuit,


## ezkl.float_to_felt(e, 7)
「浮動小数点数（少数）を、暗号分野の計算で扱える『大きな整数（有限体の元：Field Element）』に変換する関数」です。  
具体的に分解すると、以下のような処理を行っています。  
**1. float_to_felt の意味**  
- float: 通常の浮動小数点数（0.0 や 2.0 など）  
- to_felt: Field Element（有限体の元）へ変換する、という意味です。  
- 暗号資産やゼロ知識証明の回路（Circuit）の中では、通常の小数（1.5 など）をそのまま計算することができません。すべての計算は、特定の非常に大きな素数で割った余り（有限体）の整数として扱う必要があります。  

**2. 引数の 7（スケール因子）の意味**   
- これは Scale（スケール） を指定しています。内部的には $2^7 = 128$ を表します。  
- コンピュータで小数を整数に直すとき、よく「100倍して整数にする（1.50 → 150）」というようなことをしますが、これの2進数（ビット）版です。  
- e という数値に $2^7$ （128）を掛け算して、四捨五入などで整数（Felt）に変換しています。

**※そもそもなぜ「拡大（スケール）」が必要なのか？**    
有限体の世界（$0$ から $p-1$ までの整数の世界）では、0.5 や 1.25 といった「小数」をそのまま保存したり計算したりできません。   
そこで、「あらかじめ全員を $2^7$ （128）倍して、小数点を右に7マスずらし、無理やり整数にしてから有限体に入れる」 というルールを作っています。  
これが固定小数点数（Fixed-point representation）の考え方です。  
$p$ だけ指定した場合： 0.5 をどう整数にしていいかルールがありません。  
$2^7$ を指定した場合： 0.5 $\times 128 = 64$ という整数に変換できるため、有限体の世界で 64 として扱えるようになります。

In [57]:
ezkl.float_to_felt(2,7)

'0001000000000000000000000000000000000000000000000000000000000000'

**Poseidonハッシュとは？**
ブロックチェーンやゼロ知識証明（ZKP）の回路の中で計算するためにめちゃくちゃ最適化されたハッシュ関数です（Web3の世界では、一般的なSHA-256などを使うよりも、Poseidonを使った方が証明の作成スピードが圧倒的に速くなります）。

**なぜ [ ]（リスト）で囲んでいるの？**
Poseidonハッシュは、複数のデータをまとめて1つのハッシュ値にガッチャンコするのが得意な関数です。そのため、引数が1個だけであっても、ルールとしてリスト [ ] の形にして渡す必要があります。

In [58]:
ezkl.poseidon_hash([
    ezkl.float_to_felt(1,7)
])

['7b20c71000874d32c5709186347ce5f6b9527abf1fb87c0d4a0e15533bedf425']

##Setting
- 入力データはハッシュ化して絶対秘密
- 対象のセットや結果はみんなに見える状態にする
- 1件ずつ $2^{11}$ のサイズで計算する

In [59]:
run_args = ezkl.PyRunArgs()
run_args.input_visibility = "hashed/private/0"
run_args.ignore_range_check_inputs_outputs = True
run_args.param_visibility = "fixed"
run_args.output_visibility = "fixed"
run_args.variables = [("batch_size",1)]
run_args.scale_rebase_multiplier = 1000
run_args.logrows = 11

res  = ezkl.gen_settings(model_path,py_run_args=run_args)
assert res == True

DEBUG:tract_onnx.model:ONNX operator set version: 14
DEBUG:ezkl.graph.model:set batch_size to 1
DEBUG:tract_hir.infer.analyser:  Refined 2/0>: ..,? -> batch_size,9,F32
DEBUG:tract_hir.infer.analyser:  Refined 3/0>: ..,? -> batch_size,F32
DEBUG:tract_hir.infer.analyser:  Refined 4/0>: ..,? -> 1,9,F32
DEBUG:tract_core.optim:applying patch #0: declutter/0 >> declutter #5 "Identity_2" Identity
DEBUG:tract_core.optim.change_axes:  Considering change AxisChange { outlet: 0/0>, op: Rm(0) }
DEBUG:tract_core.optim.change_axes:    Change AxisChange { outlet: 0/0>, op: Rm(0) } blocked by locked interface 0/0>
DEBUG:tract_core.optim.change_axes:  Considering change AxisChange { outlet: 0/0>, op: Rm(1) }
DEBUG:tract_core.optim.change_axes:    Change AxisChange { outlet: 0/0>, op: Rm(1) } blocked by locked interface 0/0>
DEBUG:tract_core.optim.change_axes:  Considering change AxisChange { outlet: 1/0>, op: Rm(0) }
DEBUG:tract_core.optim.change_axes:    Change AxisChange { outlet: 1/0>, op: Rm(0) } b

### run_args = ezkl.PyRunArgs()意味:    

 **1. run_args.input_visibility = "hashed/private/0":**    
 今回の「セット（集合）の中に、自分の秘密のデータが含まれているか」をチェックするにあたり、入力するデータそのものは検証者（第3者）には絶対に見せない（private）ようにします。  
 ただし、改ざんを防ぐためにハッシュ値（暗号化されたデータ）に変換してから回路に投入するという特殊な設定にしています。    
 **2.run_args.ignore_range_check_inputs_outputs = True:**    
 入出力データの「値の範囲チェック」をスキップします。ezklは通常、数値が大きすぎたり小さすぎたりしないかチェックする機構が入りますが、今回はすでに Felt（有限体の元という、暗号用の特殊な整数フォーマット）に変換済みのデータを使うため、この範囲チェックをオフ（True）にして計算を効率化しています。  
 **3. run_args.param_visibility = "fixed":**  
 モデルのパラメータ（今回は所属をチェックしたい「集合・セット」のデータ）の扱いを固定（fixed）にします。誰でも検証できるように、チェック対象となるセット（例：[0.0, 2.0, 3.0... など]）の内容を固定して公開状態にしています。  
 **4. run_args.output_visibility = "fixed":**  
 計算結果（出力）の扱いを固定・公開にします。「所属しているかどうか」の結果は、検証者が確認できるように公開（fixed）しておく必要があります。  
 **5. run_args.variables = [("batch_size", 1)]:**  
 1回に処理するデータ数（バッチサイズ）を 1 に固定します。動的にサイズが変わると証明の回路が複雑になってしまうため、「今回は1件ずつ処理する回路ですよ」と教えてあげています。  
 **6. run_args.scale_rebase_multiplier = 1000:**   
 スケールの再調整（リベース）を行わないようにする設定です。小数を整数に変換する際のスケール（桁数）が計算の途中で勝手に変わると、ハッシュ値がズレてしまいます。それを防ぐために十分大きな値（1000）を安全策として設定し、自動調整を実質無効化しています。
 **7.run_args.logrows = 11:**  
 証明回路の「計算スペースの大きさ（行数）」の指定です。内部の計算テーブルの行数を $2^{11} = 2048$ 行に設定しています。この数字（11）が大きすぎると証明を作るのに時間がかかり、小さすぎると「計算が枠に収まりきらない」というエラーになります。今回の処理にちょうどいいサイズとして 11 が指定されています。

In [60]:
res = ezkl.compile_circuit(model_path,compiled_model_path,settings_path)

DEBUG:tract_onnx.model:ONNX operator set version: 14
DEBUG:ezkl.graph.model:set batch_size to 1
DEBUG:tract_hir.infer.analyser:  Refined 2/0>: ..,? -> batch_size,9,F32
DEBUG:tract_hir.infer.analyser:  Refined 3/0>: ..,? -> batch_size,F32
DEBUG:tract_hir.infer.analyser:  Refined 4/0>: ..,? -> 1,9,F32
DEBUG:tract_core.optim:applying patch #0: declutter/0 >> declutter #5 "Identity_2" Identity
DEBUG:tract_core.optim.change_axes:  Considering change AxisChange { outlet: 0/0>, op: Rm(0) }
DEBUG:tract_core.optim.change_axes:    Change AxisChange { outlet: 0/0>, op: Rm(0) } blocked by locked interface 0/0>
DEBUG:tract_core.optim.change_axes:  Considering change AxisChange { outlet: 0/0>, op: Rm(1) }
DEBUG:tract_core.optim.change_axes:    Change AxisChange { outlet: 0/0>, op: Rm(1) } blocked by locked interface 0/0>
DEBUG:tract_core.optim.change_axes:  Considering change AxisChange { outlet: 1/0>, op: Rm(0) }
DEBUG:tract_core.optim.change_axes:    Change AxisChange { outlet: 1/0>, op: Rm(0) } b

- **low scale values (<8) may impact precision という警告:**  
「スケール因子を 7 にしてるけど、精度がガタ落ちするかもしれないよ！」と、ezklがちょっと心配してくれています。前のセルで「7マスずらして無理やり整数にする」という設定（$2^7 = 128$ 倍）にしていたのを、ちゃんと見抜いてツッコんでくれているのが健気です。
- **tract_core たちの最適化の試行錯誤:**   
 ログの中に Considering change AxisChange... とか blocked by locked interface みたいな文字がたくさん並んでいます。これは、ONNXの計算グラフを「ゼロ知識証明の回路（行列の形）」に落とし込むために、内部の最適化エンジン（tract）が「ここの軸（次元）を削ったらもっと計算が軽くなるかも？ あ、でもこっちの計算ブロックにロックされてるから無理だったわ！」と、裏でめちゃくちゃ試行錯誤しながらパズルを解いている形跡。きれいに整列される opkind テーブル:最終的に、人間が書いたPyTorchのコード（SUB：引き算、PROD：掛け算など）が、すべて綺麗に整理されて idx 0 から idx 4 までの綺麗な表にマッピングされる瞬間は、見ていてどこかスッキリします。  
- **GraphModelの表について**
各ステップ（行）の具体的な流れ今回の回路は、diff = (x - y) して torch.prod(diff) する（差のすべての要素を掛け合わせる） という計算を綺麗にトレースしています。


1.   **0. idx 0 │ Input │ 7 │ │ [1, 1]:**   
    1つ目の入力データ（コード上の x）を受け取っています。
    - 状態: スケール7（128倍）の、[1, 1]（要素数1つ）のデータです。
2.   **1. idx 1 │ Input │ 7 │ │ [1, 9]:**    
    2つ目の入力データ（所属をチェックしたい集合 y）を受け取っています。
    - 状態: スケール7（128倍）の、[1, 9]（9個の要素が並んだリスト）のデータです
3.   **2. idx 2 │ SUB │ 7 │ [(0, 0), (1, 0)] │ [1, 9]:**   
    引き算（Subtraction）を実行しています（コードの x - y）。
    - 中身: inputs を見ると (0,0) と (1,0) になっているので、「ステップ0（x）」から「ステップ1（yの9個の要素）」をそれぞれ引き算しています。
     - 結果: 引き算してもスケールは変わらず 7 のまま、形は [1, 9]（9個の引き算結果）になります。    
4.   **3. idx 3 │ PROD │ 63 │ [(2, 0)] │ [1, 1]:**    
    要素の総乗（Product：掛け算）を計算しています（コードの torch.prod）。ステップ2で計算した9個の引き算結果 [(2, 0)] を、ぜんぶ掛け合わせて1つの数字にしています。
     - 注目ポイント: スケール（out_scale）が 63 に跳ね上がっています。これは、スケール7の数を9個掛け算したため、内部のビット数が $7 \times 9 = 63$ に拡大したことを正しく示しています（数学的にめちゃくちゃ正確に追跡されています！）。    
5.    **4. idx 4 │ RESHAPE │ 63 │ [(3, 0)] │ [1]:**    
    データの形を最終出力用に整えています。 ステップ3の [1, 1]（2次元の行列）という形を、[1]（ただの1次元の値）にフラット化（Reshape）して、最終的な計算結果として出力します。   


まとめこの表は、「入力された1つの値（x）から、9つの値（y）をそれぞれ引き算し、その9つの結果をすべて掛け算して、1つの最終結果を出力する回路」 が、寸分の狂いもなく暗号回路のパーツ（ゲート）にマッピングされたことを示しています。特に掛け算（PROD）によってスケールが 63 に綺麗に累積しているあたり、EZKLが裏側で完璧に固定小数点数の計算をシミュレートできていることが分かります。

ただのプログラムのビルドというより、「計算を、暗号の世界の数学パズル（多項式）に翻訳する作業」のドタバタ劇がこのログに詰まっていると思うと、なかなか愛着が湧いてくる

### SRS  
「計算の土台（数学的な舞台装置）を準備している」。

ZK-SNARKs（特にEZKLが裏で使っているKZGコミットメントという仕組み）では、証明を作成したり検証したりする前に、SRS（Structured Reference String：構造化参照文字列） という共通の数学的パラメータが必要になります
- 巨大な宇宙（パラメータ）のダウンロード  
    ログを見ると SRS does not exist, downloading... から始まって、kzg11.srs というファイルをダウンロードしています。これは、暗号的な計算を行うための「巨大な素数の波（楕円曲線上の点）」が詰まったデータです。
- 計算スペース（サイズ）の確保  
直前のセル（run_args.logrows = 11）で、「今回の計算は $2^{11} = 2048$ 行のテーブルで行うよ」と指定しました。get_srs はそのサイズ（len = 262404 などのデータ量）にぴったり合う数学的な土台をローカルに持ってくる（あるいは生成する）仕事をしています。

例えるなら…
- compile_circuit（前のステップ）:  
「引き算して、全部掛け算する」という計算のルール（プラモデルの設計図）を作った状態。

- get_srs（このステップ）:  
そのプラモデルを組み立てて安全に展示するための、寸分の狂いもない「専用の作業台（ベース）」を引っ張ってきた状態。

このSRSという土台（作業台）がしっかり固定されているからこそ、次のステップ（gen_witness や prove）で「私は秘密の入力 X を使って、この設計図通りに正しく計算しました！」という証拠（証明）を、嘘偽りなくガッチャンコと生成できるようになります。

★裏側で kzg.ezkl.xyz という暗号学の専門サーバーに通信しにいっている点も、Web3・ cryptographicな雰囲気が出ていてワクワクするポイント！

In [61]:
res = await ezkl.get_srs(settings_path)

INFO:ezkl.execute:SRS already exists at that path
INFO:ezkl.execute:read 262404 bytes from file (vector of len = 262404)
INFO:ezkl.execute:file hash: 257fa566ed9bc0767d3e63e92b5e966829fa3347d320a32055dc31ee7d33f8a4


### Witness
コンパイル（compile_circuit）でできあがった数式の枠組み（空のマス目だらけの回路）に対して、実際の入力データを流し込み、「すべてのマスの計算結果（証拠：Witness）をパチパチと埋めていく瞬間」がリアルタイムに可視化されています。


In [62]:
res = ezkl.gen_witness(data_path,compiled_model_path,witness_path)
assert os.path.isfile(witness_path)

DEBUG:ezkl.graph.model:calculating num of constraints using dummy model layout...
DEBUG:ezkl.graph.model:laying out 0: Input
DEBUG:ezkl.circuit.ops.region:(rows=0, coord=0, constants=0, max_lookup_inputs=0, min_lookup_inputs=0, max_range_size=0, dynamic_lookup_col_coord=0, shuffle_col_coord=0, max_dynamic_input_len=0)
DEBUG:ezkl.circuit.ops:constraining input to be identity
DEBUG:ezkl.graph.model:------------ output node 0: "Tensor { inner: [-149319993740704887712865435952675098936], dims: [1, 1], scale: None, visibility: None }"
DEBUG:ezkl.graph.model:------------ layout of 0 took 5.47949ms
DEBUG:ezkl.graph.model:laying out 1: Input
DEBUG:ezkl.circuit.ops.region:(rows=0, coord=1, constants=0, max_lookup_inputs=0, min_lookup_inputs=0, max_range_size=0, dynamic_lookup_col_coord=0, shuffle_col_coord=0, max_dynamic_input_len=0)
DEBUG:ezkl.circuit.ops:constraining input to be identity
DEBUG:ezkl.graph.model:------------ output node 1: "Tensor { inner: [-149319993740704887712865435952675098


#### gen_witness のログが面白い理由
- laying out 0: Input 〜 laying out 4: RESHAPE の連動    
    直前のステップで作られた idx 0 から idx 4 までの設計図通りに、上から順番にデータが処理されているのが分かります。「今、引き算（SUB）のマス目を埋めたぞ！」「よし、次は掛け算（PROD）のマス目を埋めるぞ！」と、EZKLが猛スピードで電卓を叩いている様子が伝わってきます。

- 謎の巨大な数字の出現:
    ログの inner: [-14931999374070488771...] のような、目がチカチカするほど巨大な数字。
    一見「えっ、バグってめちゃくちゃな値になってる？」と思ってしまいますが、これこそが有限体（Finite Field）の魔法です。

    私たちが普段使う「マイナスの値」や「拡大された整数」が、暗号の世界の超巨大な素数（ポセイドンハッシュ等でも使われる値）で割った余りの世界に翻訳された結果、このような「暗号の姿」になってマス目に書き込まれています。

- ミリ秒（ms）やマイクロ秒（µs）単位の職人技:    
    各レイアウトの最後に took 4.062566ms や took 534.462µs（100万分の1秒！）のように、かかった時間が細かく記録されています。「引き算はちょっと時間がかかったけど、形を整える（RESHAPE）のは一瞬で終わったぜ」という報告に愛着が湧きますよね。

### データ準備  
- 間違ったデータ

In [63]:
data_path_faulty = os.path.join('input_faulty.json')

witness_path_faulty = os.path.join('witness_faulty.json')

x = torch.ones(1,*[1], requires_grad=True)
y = torch.tensor([0.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0], requires_grad=True)

y = y.unsqueeze(0)
y = y.reshape(1, 9)

data_array_x = ((x).detach().numpy()).reshape([-1]).tolist()
data_array_y = result
print(data_array_y)

data = dict(input_data = [data_array_x, data_array_y])

print(data)

    # Serialize data into file:
json.dump( data, open(data_path_faulty, 'w' ))

res = ezkl.gen_witness(data_path_faulty, compiled_model_path, witness_path_faulty)
assert os.path.isfile(witness_path_faulty)
res

DEBUG:ezkl.graph.model:calculating num of constraints using dummy model layout...
DEBUG:ezkl.graph.model:laying out 0: Input
DEBUG:ezkl.circuit.ops.region:(rows=0, coord=0, constants=0, max_lookup_inputs=0, min_lookup_inputs=0, max_range_size=0, dynamic_lookup_col_coord=0, shuffle_col_coord=0, max_dynamic_input_len=0)
DEBUG:ezkl.circuit.ops:constraining input to be identity
DEBUG:ezkl.graph.model:------------ output node 0: "Tensor { inner: [-65539363029252089940147629258685603718], dims: [1, 1], scale: None, visibility: None }"
DEBUG:ezkl.graph.model:------------ layout of 0 took 3.321547ms
DEBUG:ezkl.graph.model:laying out 1: Input
DEBUG:ezkl.circuit.ops.region:(rows=0, coord=1, constants=0, max_lookup_inputs=0, min_lookup_inputs=0, max_range_size=0, dynamic_lookup_col_coord=0, shuffle_col_coord=0, max_dynamic_input_len=0)
DEBUG:ezkl.circuit.ops:constraining input to be identity
DEBUG:ezkl.graph.model:------------ output node 1: "Tensor { inner: [-149319993740704887712865435952675098

['c9ee727ef8f6307295e9e69831ecddb7866fa7c072dc626ad45a803258c2d411', '59a711784a1a751ed727c22a8ffb8f9f27171bd2d8f57fca2ba9ebf3dac3f023', '2ad51066722d0ce27aa8b16c8da8b229d009e8f804a49771309b3347f23ff504', '5f8c61fda484271ae4b14dd122a938ba74fa1cf3b1a4483ba4253b77c87a920a', 'a761d7c193856b47c39447bdd4a2a85fca8aebc604d8555299e01a5b5c39cc2c', '28c2facf747bda1cce78c265233ac09dda1708937c90af3b761ca579c248010d', 'e86709c7126a9bf640640b5eb68833e2e11f20be881887716d567ef8310db70a', '7b5892a20a6dfc7b96f2ca5482608a9d349b120c571b1d338c2b807e1d955400', 'dbe3f2ef34b1f0dc9e98d3e5af291c1013511e70f935e30ff113f3df710ca42f']
{'input_data': [[1.0], ['c9ee727ef8f6307295e9e69831ecddb7866fa7c072dc626ad45a803258c2d411', '59a711784a1a751ed727c22a8ffb8f9f27171bd2d8f57fca2ba9ebf3dac3f023', '2ad51066722d0ce27aa8b16c8da8b229d009e8f804a49771309b3347f23ff504', '5f8c61fda484271ae4b14dd122a938ba74fa1cf3b1a4483ba4253b77c87a920a', 'a761d7c193856b47c39447bdd4a2a85fca8aebc604d8555299e01a5b5c39cc2c', '28c2facf747bda1cce78c2

{'inputs': [['8000000000000000000000000000000000000000000000000000000000000000'],
  ['c9ee727ef8f6307295e9e69831ecddb7866fa7c072dc626ad45a803258c2d411',
   '59a711784a1a751ed727c22a8ffb8f9f27171bd2d8f57fca2ba9ebf3dac3f023',
   '2ad51066722d0ce27aa8b16c8da8b229d009e8f804a49771309b3347f23ff504',
   '5f8c61fda484271ae4b14dd122a938ba74fa1cf3b1a4483ba4253b77c87a920a',
   'a761d7c193856b47c39447bdd4a2a85fca8aebc604d8555299e01a5b5c39cc2c',
   '28c2facf747bda1cce78c265233ac09dda1708937c90af3b761ca579c248010d',
   'e86709c7126a9bf640640b5eb68833e2e11f20be881887716d567ef8310db70a',
   '7b5892a20a6dfc7b96f2ca5482608a9d349b120c571b1d338c2b807e1d955400',
   'dbe3f2ef34b1f0dc9e98d3e5af291c1013511e70f935e30ff113f3df710ca42f']],
 'outputs': [['d678720887c1ac81b4d97ec93949ece23846ae676453e945645d73ae1b1b480f'],
  ['c9ee727ef8f6307295e9e69831ecddb7866fa7c072dc626ad45a803258c2d411',
   '59a711784a1a751ed727c22a8ffb8f9f27171bd2d8f57fca2ba9ebf3dac3f023',
   '2ad51066722d0ce27aa8b16c8da8b229d009e8f804a49771

- 正しいデータ

In [64]:
# now generate a truthy input + witness file (x input not in the set)
import random

# Generate a random integer between 1 and 8, inclusive
random_value = random.randint(1, 8)

data_path_truthy = os.path.join('input_truthy.json')

witness_path_truthy = os.path.join('witness_truthy.json')

set = [0.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0]

x = torch.tensor([set[random_value]])
y = torch.tensor(set, requires_grad=True)

y = y.unsqueeze(0)
y = y.reshape(1, 9)

x = x.unsqueeze(0)
x = x.reshape(1,1)

data_array_x = ((x).detach().numpy()).reshape([-1]).tolist()
data_array_y = result
print(data_array_y)

data = dict(input_data = [data_array_x, data_array_y])

print(data)

# Serialize data into file:
json.dump( data, open(data_path_truthy, 'w' ))

res = ezkl.gen_witness(data_path_truthy, compiled_model_path, witness_path_truthy)
assert os.path.isfile(witness_path_truthy)

DEBUG:ezkl.graph.model:calculating num of constraints using dummy model layout...
DEBUG:ezkl.graph.model:laying out 0: Input
DEBUG:ezkl.circuit.ops.region:(rows=0, coord=0, constants=0, max_lookup_inputs=0, min_lookup_inputs=0, max_range_size=0, dynamic_lookup_col_coord=0, shuffle_col_coord=0, max_dynamic_input_len=0)
DEBUG:ezkl.circuit.ops:constraining input to be identity
DEBUG:ezkl.graph.model:------------ output node 0: "Tensor { inner: [1987357085314706264469848738151585065], dims: [1, 1], scale: None, visibility: None }"
DEBUG:ezkl.graph.model:------------ layout of 0 took 3.180973ms
DEBUG:ezkl.graph.model:laying out 1: Input
DEBUG:ezkl.circuit.ops.region:(rows=0, coord=1, constants=0, max_lookup_inputs=0, min_lookup_inputs=0, max_range_size=0, dynamic_lookup_col_coord=0, shuffle_col_coord=0, max_dynamic_input_len=0)
DEBUG:ezkl.circuit.ops:constraining input to be identity
DEBUG:ezkl.graph.model:------------ output node 1: "Tensor { inner: [-14931999374070488771286543595267509893

['c9ee727ef8f6307295e9e69831ecddb7866fa7c072dc626ad45a803258c2d411', '59a711784a1a751ed727c22a8ffb8f9f27171bd2d8f57fca2ba9ebf3dac3f023', '2ad51066722d0ce27aa8b16c8da8b229d009e8f804a49771309b3347f23ff504', '5f8c61fda484271ae4b14dd122a938ba74fa1cf3b1a4483ba4253b77c87a920a', 'a761d7c193856b47c39447bdd4a2a85fca8aebc604d8555299e01a5b5c39cc2c', '28c2facf747bda1cce78c265233ac09dda1708937c90af3b761ca579c248010d', 'e86709c7126a9bf640640b5eb68833e2e11f20be881887716d567ef8310db70a', '7b5892a20a6dfc7b96f2ca5482608a9d349b120c571b1d338c2b807e1d955400', 'dbe3f2ef34b1f0dc9e98d3e5af291c1013511e70f935e30ff113f3df710ca42f']
{'input_data': [[3.0], ['c9ee727ef8f6307295e9e69831ecddb7866fa7c072dc626ad45a803258c2d411', '59a711784a1a751ed727c22a8ffb8f9f27171bd2d8f57fca2ba9ebf3dac3f023', '2ad51066722d0ce27aa8b16c8da8b229d009e8f804a49771309b3347f23ff504', '5f8c61fda484271ae4b14dd122a938ba74fa1cf3b1a4483ba4253b77c87a920a', 'a761d7c193856b47c39447bdd4a2a85fca8aebc604d8555299e01a5b5c39cc2c', '28c2facf747bda1cce78c2

In [65]:
witness = json.load(open(witness_path, "r"))
witness

{'inputs': [['0000000000000000000000000000000000000000000000000000000000000000'],
  ['c9ee727ef8f6307295e9e69831ecddb7866fa7c072dc626ad45a803258c2d411',
   '59a711784a1a751ed727c22a8ffb8f9f27171bd2d8f57fca2ba9ebf3dac3f023',
   '2ad51066722d0ce27aa8b16c8da8b229d009e8f804a49771309b3347f23ff504',
   '5f8c61fda484271ae4b14dd122a938ba74fa1cf3b1a4483ba4253b77c87a920a',
   'a761d7c193856b47c39447bdd4a2a85fca8aebc604d8555299e01a5b5c39cc2c',
   '28c2facf747bda1cce78c265233ac09dda1708937c90af3b761ca579c248010d',
   'e86709c7126a9bf640640b5eb68833e2e11f20be881887716d567ef8310db70a',
   '7b5892a20a6dfc7b96f2ca5482608a9d349b120c571b1d338c2b807e1d955400',
   'dbe3f2ef34b1f0dc9e98d3e5af291c1013511e70f935e30ff113f3df710ca42f']],
 'pretty_elements': {'rescaled_inputs': [['0'],
   ['-1166562451099257000000000000000000000',
    '1239501670124479900000000000000000000',
    '15526227229021144000000000000000000',
    '-1142112490839209800000000000000000000',
    '575887733162474800000000000000000000',
    '

### 制約の設定　witnessの出力を０に固定する★主役

通常の「AIモデルの推論を通ったことの証明（zkML）」と、今回の「メンバーに含まれていることの証明（セット・メンバーシップ）」では、何を証明のゴール（制約）するかの設計が根本的に異なります。その違いを整理すると、以下のようになります。    
**1. 通常のモデル推論証明の場合**     
        - やること:   
        入力 $x$ を入れて、モデルのレイヤー（Linear, ReLUなど）を順番に計算するだけ。  
        - 制約の対象:   
        「すべてのパズル（計算）の途中経過が、AIのルール通りに正しく計算されていること」   
        - ゴール:  
            出力された結果（例: 0.98 という推論スコア）が、改ざんされずにモデルから正しく導き出されたことさえ分かればOK。    
**2. メンバーシップ証明（今回）の場合**   
        - やること: 入力 $x$ が 集合 $y$ の中にあるかを確かめる。   
        - 問題点: ただモデル（$x - y$ の総乗）を動かすだけだと、中身が違っていても「間違った入力（例: 1.0）に対して、正しく計算した結果（例: 120）が出ました！」という、「正しく不合格になった証明」が完成してしまいます。これでは検証者は、その人がメンバーなのかそうでないのか判断できません
        - 解決策（今回のコードの処理）:だからこそ、ここで「出口に 0 というバリケード（制約）」を設ける必要があります。

In [66]:

witness = json.load(open(witness_path, "r"))
witness["outputs"][0] = ["0000000000000000000000000000000000000000000000000000000000000000"]
json.dump(witness, open(witness_path, "w"))

witness = json.load(open(witness_path, "r"))
print(witness["outputs"][0])



['0000000000000000000000000000000000000000000000000000000000000000']


In [67]:
res = ezkl.setup(
        compiled_model_path,
        vk_path,
        pk_path,
        witness_path = witness_path,
    )

assert res == True
assert os.path.isfile(vk_path)
assert os.path.isfile(pk_path)

DEBUG:ezkl.pfsys.srs:loading srs from "/root/.ezkl/srs/kzg11.srs"
DEBUG:ezkl.graph.vars:number of blinding factors: 5
DEBUG:ezkl.graph.vars:model uses 3 advice blocks (size=2)
DEBUG:ezkl.graph.vars:model uses 1 fixed columns
DEBUG:ezkl.graph.vars:model uses [] instance dims
DEBUG:ezkl.graph.model:configuring model
DEBUG:ezkl.graph:degree: 6, log2_ceil of degrees: 3.0
DEBUG:ezkl.graph:circuit size: 
 {
  "logrows": 11,
  "num_advice_columns": 9,
  "num_challenges": 0,
  "num_fixed": 5,
  "num_instances": 1,
  "num_selectors": 15
}
DEBUG:halo2_proofs.plonk.keygen:Creating domain with degree 6
DEBUG:ezkl.circuit.modules.planner:spawning module 0
DEBUG:ezkl.circuit.modules.planner:spawning module 2
INFO:ezkl.graph.model:model layout...
DEBUG:ezkl.graph.model:laying out 0: Input
DEBUG:ezkl.circuit.ops.region:(rows=0, coord=0, constants=0, max_lookup_inputs=0, min_lookup_inputs=0, max_range_size=0, dynamic_lookup_col_coord=0, shuffle_col_coord=0, max_dynamic_input_len=0)
DEBUG:ezkl.graph.mod

###証明
正しくても正しくなくても証明は通ります


In [68]:
# GENERATE A PROOF


proof_path = os.path.join('test.pf')

res = ezkl.prove(
    witness_path,
    compiled_model_path,
    pk_path,
    proof_path
)

print(res)
assert os.path.isfile(proof_path)

DEBUG:ezkl.graph:rescaled and processed public inputs: {
  "inputs": [],
  "outputs": [],
  "processed_inputs": [
    [
      "0x11d4c25832805ad46a62dc72c0a76f86b7ddec3198e6e9957230f6f87e72eec9"
    ]
  ],
  "processed_outputs": [],
  "processed_params": [],
  "rescaled_inputs": [],
  "rescaled_outputs": []
}
DEBUG:ezkl.graph:public inputs: [0x11d4c25832805ad46a62dc72c0a76f86b7ddec3198e6e9957230f6f87e72eec9]
DEBUG:ezkl.pfsys:loading proving key from "test.pk"
DEBUG:ezkl.graph.vars:number of blinding factors: 5
DEBUG:ezkl.graph.vars:model uses 3 advice blocks (size=2)
DEBUG:ezkl.graph.vars:model uses 1 fixed columns
DEBUG:ezkl.graph.vars:model uses [] instance dims
DEBUG:ezkl.graph.model:configuring model
DEBUG:ezkl.graph:degree: 6, log2_ceil of degrees: 3.0
DEBUG:ezkl.graph:circuit size: 
 {
  "logrows": 11,
  "num_advice_columns": 9,
  "num_challenges": 0,
  "num_fixed": 5,
  "num_instances": 1,
  "num_selectors": 15
}
DEBUG:halo2_proofs.plonk.keygen:Creating domain with degree 6
INFO

{'instances': [['c9ee727ef8f6307295e9e69831ecddb7866fa7c072dc626ad45a803258c2d411']], 'proof': '0x03f59cc86b1ac46c18dfc4cd721c1310e854e45c40fb3fe405a8c324bdfed49800845d7cdc4ada5d18091ea824e570e3e45a3211b6271578c89246ad8b9ee843300e9cf04422689b8cbd3f35e84a580dcf4db09a3af0cce1187a3755b0510c6e0608b3b9685f881d0141e44623dcbcd549c3b5607a81ab6fa50d29c9de7332230daa1e7125c56e07973e518848052f9687310e84c5949dbdc48c77a0d50820dd0fe6f49400d6e99e7b812396df620bd0aa12f276aa6c891d8dfb6845e2da62a520f1085eecfa86fb7f8c290213cb4e7955089a0e07b72485916f933717b95b380c36cbcebf27e5a9a3b31e924dc07262273120b0b924429b22ab1a47ec07e72d053dcaaff15c96d68c37eb8f1311602922a148afbec2c37d57dc646029d23e0013934e4448f187251ca68aa8527af2597bd2b5970edcfff0e007d1a1d087931a2e7d788421ea46aea4f4aba8630dd0d7fa86289652148c9d104185a77ce0c8820f23051471a2351368507f191c4a1d8e2479612ff45d3d4807dd2aa03ade628206cf1f5fc120c7dea2842770418acd7ec9c7a0459503789e5d20b480660b893c1c2dd5307670906853ce016c11f38ec44c99cd18e0ce13added7611acb89c8ab056d32

- 正しくない場合

In [69]:
proof_path_faulty = os.path.join('test_faulty.pf')

res = ezkl.prove(
    witness_path_faulty,
    compiled_model_path,
    pk_path,
    proof_path_faulty
)
print(res)
assert os.path.isfile(proof_path_faulty)


DEBUG:ezkl.graph:rescaled and processed public inputs: {
  "inputs": [],
  "outputs": [],
  "processed_inputs": [
    [
      "0x25f4ed3b53150e4a0d7cb81fbf7a52b9f6e57c34869170c5324d870010c7207b"
    ]
  ],
  "processed_outputs": [],
  "processed_params": [],
  "rescaled_inputs": [],
  "rescaled_outputs": []
}
DEBUG:ezkl.graph:public inputs: [0x25f4ed3b53150e4a0d7cb81fbf7a52b9f6e57c34869170c5324d870010c7207b]
DEBUG:ezkl.pfsys:loading proving key from "test.pk"
DEBUG:ezkl.graph.vars:number of blinding factors: 5
DEBUG:ezkl.graph.vars:model uses 3 advice blocks (size=2)
DEBUG:ezkl.graph.vars:model uses 1 fixed columns
DEBUG:ezkl.graph.vars:model uses [] instance dims
DEBUG:ezkl.graph.model:configuring model
DEBUG:ezkl.graph:degree: 6, log2_ceil of degrees: 3.0
DEBUG:ezkl.graph:circuit size: 
 {
  "logrows": 11,
  "num_advice_columns": 9,
  "num_challenges": 0,
  "num_fixed": 5,
  "num_instances": 1,
  "num_selectors": 15
}
DEBUG:halo2_proofs.plonk.keygen:Creating domain with degree 6
INFO

{'instances': [['7b20c71000874d32c5709186347ce5f6b9527abf1fb87c0d4a0e15533bedf425']], 'proof': '0x115753018d20a16ae3e2cc3d58ffa50025ac26887d4e7e20685ef1c316e5c1232639e2993943d3c66d9fdfefc6999813fcbd67665af08cf023e59708e38884512539c14e735e18f95cfb9680e9c79a38c81642889ac797a3de7d0003bbd0d0572689928332f1c509099c8b13c601e3b0a30108af1938c6be1464478b8799aef4028986643aaa2b9b835bdb974ffe9b46a7258c4ee6a7338014d7a1b473ea50d922ee1182311d173a47b427fb8fb1c4a129f81f1323df18d6b7802dde53acc6db236cf431e908d90f9006bd9fdc2452c213cbd2b9a36a87d4fc02a350c834443e2b9fe7eb8c44d0c957b78d9915539fd69fb4d8a7b66ef3bf8a2105cb6d6c088c151e616f2488feee0cb6bfea2d1b1950b647cb3801747dc30971b9e05d179ff21ea0e9f36173afd6debcb295d06a195813bd9809bcb665467b957d25e3b53a681493646c33617275d59c476475513bafd5748ebd6e15699de60415e8b787012c27a33f4dcf719aa2ce63e97876603eb20adee14ca593298e4cbd3eb4e1dbe82c0f689a1284d752a2bb95333b398170aff3eaf228a2b9ab9412f53afe10b5db7415b8faa27f6b8d89e55bd4f5768530b9564292674df77ad293d7f80bc410bbfb271b69

- 正しい場合

In [70]:
proof_path_truthy = os.path.join("test_truthy.pf")

res = ezkl.prove(
    witness_path_truthy,
    compiled_model_path,
    pk_path,
    proof_path_truthy
)
print(res)
assert os.path.isfile(proof_path_truthy)

DEBUG:ezkl.graph:rescaled and processed public inputs: {
  "inputs": [],
  "outputs": [],
  "processed_inputs": [
    [
      "0x04f53ff247339b307197a404f8e809d029b2a88d6cb1a87ae20c2d726610d52a"
    ]
  ],
  "processed_outputs": [],
  "processed_params": [],
  "rescaled_inputs": [],
  "rescaled_outputs": []
}
DEBUG:ezkl.graph:public inputs: [0x04f53ff247339b307197a404f8e809d029b2a88d6cb1a87ae20c2d726610d52a]
DEBUG:ezkl.pfsys:loading proving key from "test.pk"
DEBUG:ezkl.graph.vars:number of blinding factors: 5
DEBUG:ezkl.graph.vars:model uses 3 advice blocks (size=2)
DEBUG:ezkl.graph.vars:model uses 1 fixed columns
DEBUG:ezkl.graph.vars:model uses [] instance dims
DEBUG:ezkl.graph.model:configuring model
DEBUG:ezkl.graph:degree: 6, log2_ceil of degrees: 3.0
DEBUG:ezkl.graph:circuit size: 
 {
  "logrows": 11,
  "num_advice_columns": 9,
  "num_challenges": 0,
  "num_fixed": 5,
  "num_instances": 1,
  "num_selectors": 15
}
DEBUG:halo2_proofs.plonk.keygen:Creating domain with degree 6
INFO

{'instances': [['2ad51066722d0ce27aa8b16c8da8b229d009e8f804a49771309b3347f23ff504']], 'proof': '0x08e67f106bea1c75476d8900d3eed7f0882c8227b7083036e69d73f2e7e0590800ecda461422e47818c215d26ab054bd4fa4a4186f0e7e74407f5534b0ae884e173f38e6b26c3db1c5829936c154d9209197fb1f7e6386507d63b83438cc44b624bdbd0fb9bf95c23d268ecceb7c41c64e91f3e2323295fe53fe7fe373263c2b2e44c7d9ca0061b4fc322e35f905cd059b04dcd8afb5230dd8877807ee445c111bad1386f0cd4691a0a63b4b603a690f3d2371216024bf88c889d0d577ab836413604fa2ab8b68d03012c5cccb24079fb7901f15c9b9cac6d699231b68bf61090266cb9e64af4c3446483623b80a457abc2c311c4b7f3be16adccfd0270fbae10ec19dc5f9daed552a3e2bc5643dafd51c989e95e41cb260948b2648cd728fd32ede815c427578d3195dd39fba75f6458a7a9caba8f30409301c94c1428827db2ee22657865b74851b02ebf83fd38f3f4d200671e0b28d2ceeaa8c0d29b70380022566576cba41877fa05a400892cccd6c03a8fc4a1f865031b7d20185d3be6106950dc3b5ed04183d6ec3ee81c150964e11e8bd585d7fc80c7c651ad55ca73e003b02b015f87cd3d12d2cb9ba06bb1e5e91a75c74efe074dabe7c1a43133a9b0d8ce9

### 検証

In [71]:
res = ezkl.verify(
    proof_path,
    settings_path,
    vk_path
)

DEBUG:ezkl.pfsys.srs:loading srs from "/root/.ezkl/srs/kzg11.srs"
DEBUG:ezkl.pfsys:loading verification key from "test.vk"
DEBUG:ezkl.graph.vars:number of blinding factors: 5
DEBUG:ezkl.graph.vars:model uses 3 advice blocks (size=2)
DEBUG:ezkl.graph.vars:model uses 1 fixed columns
DEBUG:ezkl.graph.vars:model uses [] instance dims
DEBUG:ezkl.graph.model:configuring model
DEBUG:ezkl.graph:degree: 6, log2_ceil of degrees: 3.0
DEBUG:ezkl.graph:circuit size: 
 {
  "logrows": 11,
  "num_advice_columns": 9,
  "num_challenges": 0,
  "num_fixed": 5,
  "num_instances": 1,
  "num_selectors": 15
}
DEBUG:halo2_proofs.plonk.keygen:Creating domain with degree 6
INFO:ezkl.pfsys:loaded verification key ✅
INFO:ezkl.execute:verify took 0.6
INFO:ezkl.execute:verified: true


#### 正しくない場合はエラー

In [72]:
res = ezkl.verify(
    proof_path_faulty,
    settings_path,
    vk_path
)

assert res == True

DEBUG:ezkl.pfsys.srs:loading srs from "/root/.ezkl/srs/kzg11.srs"
DEBUG:ezkl.pfsys:loading verification key from "test.vk"
DEBUG:ezkl.graph.vars:number of blinding factors: 5
DEBUG:ezkl.graph.vars:model uses 3 advice blocks (size=2)
DEBUG:ezkl.graph.vars:model uses 1 fixed columns
DEBUG:ezkl.graph.vars:model uses [] instance dims
DEBUG:ezkl.graph.model:configuring model
DEBUG:ezkl.graph:degree: 6, log2_ceil of degrees: 3.0
DEBUG:ezkl.graph:circuit size: 
 {
  "logrows": 11,
  "num_advice_columns": 9,
  "num_challenges": 0,
  "num_fixed": 5,
  "num_instances": 1,
  "num_selectors": 15
}
DEBUG:halo2_proofs.plonk.keygen:Creating domain with degree 6
INFO:ezkl.pfsys:loaded verification key ✅
INFO:ezkl.execute:verify took 0.7
INFO:ezkl.execute:verified: false


RuntimeError: Failed to run verify: [halo2] The constraint system is not satisfied

このセルを実行すると、ログに INFO:ezkl.execute:verified: false と表示され、その下の assert res == True でエラー（AssertionError）が発生してプログラムが完全にストップします

#### 正しい場合は通る

In [73]:
res = ezkl.verify(
    proof_path_truthy,
    settings_path,
    vk_path
)

assert res == True

DEBUG:ezkl.pfsys.srs:loading srs from "/root/.ezkl/srs/kzg11.srs"
DEBUG:ezkl.pfsys:loading verification key from "test.vk"
DEBUG:ezkl.graph.vars:number of blinding factors: 5
DEBUG:ezkl.graph.vars:model uses 3 advice blocks (size=2)
DEBUG:ezkl.graph.vars:model uses 1 fixed columns
DEBUG:ezkl.graph.vars:model uses [] instance dims
DEBUG:ezkl.graph.model:configuring model
DEBUG:ezkl.graph:degree: 6, log2_ceil of degrees: 3.0
DEBUG:ezkl.graph:circuit size: 
 {
  "logrows": 11,
  "num_advice_columns": 9,
  "num_challenges": 0,
  "num_fixed": 5,
  "num_instances": 1,
  "num_selectors": 15
}
DEBUG:halo2_proofs.plonk.keygen:Creating domain with degree 6
INFO:ezkl.pfsys:loaded verification key ✅
INFO:ezkl.execute:verify took 0.10
INFO:ezkl.execute:verified: true


#### エラーが出た場合に表示したい

In [74]:
try:
    res = ezkl.verify(
        proof_path_faulty,
        settings_path,
        vk_path,
    )
except Exception as e:
    # エラーが発生した場合、ここでキャッチして好きな文字を表示する
    print("【判定】エラー：不正な証明書のため、検証に失敗しました！")
    # 必要なら実際のエラー内容も表示できます
    # print("詳細エラー:", e)


DEBUG:ezkl.pfsys.srs:loading srs from "/root/.ezkl/srs/kzg11.srs"
DEBUG:ezkl.pfsys:loading verification key from "test.vk"
DEBUG:ezkl.graph.vars:number of blinding factors: 5
DEBUG:ezkl.graph.vars:model uses 3 advice blocks (size=2)
DEBUG:ezkl.graph.vars:model uses 1 fixed columns
DEBUG:ezkl.graph.vars:model uses [] instance dims
DEBUG:ezkl.graph.model:configuring model
DEBUG:ezkl.graph:degree: 6, log2_ceil of degrees: 3.0
DEBUG:ezkl.graph:circuit size: 
 {
  "logrows": 11,
  "num_advice_columns": 9,
  "num_challenges": 0,
  "num_fixed": 5,
  "num_instances": 1,
  "num_selectors": 15
}
DEBUG:halo2_proofs.plonk.keygen:Creating domain with degree 6
INFO:ezkl.pfsys:loaded verification key ✅
INFO:ezkl.execute:verify took 0.8
INFO:ezkl.execute:verified: false


【判定】エラー：不正な証明書のため、検証に失敗しました！


次は文字列（テキスト）を含むリストのメンバーシップ証明に挑戦したい